In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.features.feature_engineer.min_features import _detect_star_players
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.utils.helpers import *
from src.utils.dataScraper import *
from live import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{}

Out Players:
{'OKC': ['Jalen Williams'], 'LAL': ['Luka Doncic']}
Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 2 teams with confirmed lineups
Updated 0 teams with questionable players


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
p25 = pd.read_csv('data/raw/playoff_stats/P25.csv').sort_values(by='GAME_DATE')
s25 = pd.concat([s25, p25])
s25 = _detect_star_players(s25)

s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
s26 = pd.concat([s26, p26])
s26 = _detect_star_players(s26)

base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS,TEAM_SPREAD,GAME_TOTAL,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED,TOP_PLAYER,SECOND_TOP_PLAYER,THIRD_TOP_PLAYER,IS_TOP_STAR,IS_TOP_1_STAR,ACTIVE_STARS_COUNT,TOP_STAR_ACTIVE,TOP_PLAYER_ACTIVE,SECOND_PLAYER_ACTIVE,THIRD_PLAYER_ACTIVE,name
27942,NaN,NaN,NaN,2025-26,1630540,Miles McBride,Miles,1610612752,NYK,New York Knicks,42500213,2026-05-08,NYK @ PHI,W,21.028333,1,6,0.167,1,5,0.2,0,0,0.000,0,0,0,2,0,0,2,0,1,0,3,-4,12.0,0,0,10.0,1,21:02,1,116.6,114.3,114.3,128.0,126.8,126.8,-11.3,-12.5,-12.5,0.125,0.0,25.0,0.000,0.000,0.000,0.0,0.0,0.250,0.250,0.125,0.127,93.36,94.73,78.94,94.73,0.000,42,1.0,6.0,G,4.24,1.60,1.0,0.0,1.0,24.0,0.0,0.0,17.0,0.0,1.0,0.00,1.0,5.0,0.2,1.0,2.0,0.50,38,76,0.500,9,27,0.333,23,32,0.719,13,36,49,25,15.0,6,4,3,21,25,108,14.0,117.3,114.9,101.0,101.1,16.3,13.8,0.658,1.67,18.7,0.349,0.776,0.576,0.160,0.559,0.599,92.6,93.5,77.92,94,0.600,1610612755,PHI,Philadelphia 76ers,36,84,0.429,9,32,0.281,13,16,0.813,9,24,33,23,11.0,7,3,4,25,21,94,-14.0,101.0,101.1,117.3,114.9,-16.3,-13.8,0.639,2.09,18.1,0.224,0.651,0.424,0.118,0.482,0.516,92.6,93.5,77.92,93,0.400,1,SG,25.0,NaN,NaN,3.5,214.5,1,0.142665,0.095110,0.000000,0,4,Karl-Anthony Towns,Jalen Brunson,OG Anunoby,0,0,2,1,1,1,0,Miles McBride
27943,NaN,NaN,NaN,2025-26,1629656,Quentin Grimes,Quentin,1610612755,PHI,Philadelphia 76ers,42500213,2026-05-08,PHI vs. NYK,L,22.466667,2,6,0.333,2,5,0.4,0,0,0.000,1,1,2,2,1,0,0,0,4,1,6,-17,10.4,0,0,12.0,1,22:28,1,77.7,79.1,79.1,121.7,115.9,115.9,-44.0,-36.8,-36.8,0.182,2.0,22.2,0.036,0.050,0.042,11.1,11.1,0.500,0.500,0.140,0.144,91.53,92.94,77.45,92.94,0.008,43,2.0,6.0,NaN,3.97,1.63,5.0,4.0,9.0,19.0,0.0,0.0,12.0,0.0,1.0,0.00,2.0,5.0,0.4,1.0,1.0,1.00,36,84,0.429,9,32,0.281,13,16,0.813,9,24,33,23,11.0,7,3,4,25,21,94,-14.0,101.0,101.1,117.3,114.9,-16.3,-13.8,0.639,2.09,18.1,0.224,0.651,0.424,0.118,0.482,0.516,92.6,93.5,77.92,93,0.400,1610612752,NYK,New York Knicks,38,76,0.500,9,27,0.333,23,32,0.719,13,36,49,25,15.0,6,4,3,21,25,108,14.0,117.3,114.9,101.0,101.1,16.3,13.8,0.658,1.67,18.7,0.349,0.776,0.576,0.160,0.559,0.599,92.6,93.5,77.92,94,0.600,1,SG,25.0,NaN,NaN,-3.5,214.5,1,0.267062,0.089021,0.089021,1,4,Joel Embiid,Tyrese Maxey,Paul George,0,0,3,1,1,1,1,Quen

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_odds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_odds = pd.json_normalize(data)

print("Loaded:", file.name)
team_odds.head()

Loaded: NBA_20260509_143842.json


,home_team,away_team,commence_time,bookmakers
0,Cleveland Cavaliers,Detroit Pistons,2026-05-09 19:14:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Los Angeles Lakers,Oklahoma City Thunder,2026-05-10 00:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,Philadelphia 76ers,New York Knicks,2026-05-10 19:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Minnesota Timberwolves,San Antonio Spurs,2026-05-10 23:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = s26
ast_df = s26
reb_df = s26
min_df = s26


#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
# lines_dfs = lines_dfs[lines_dfs['COMMENCE_TIME'] == current_date]
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
# lines_us = lines_us[lines_us['COMMENCE_TIME'] == current_date]
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-05-09 14:34:15
US latest pull: 2026-05-09 14:38:43


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Tobias Harris,Over,21.5,-137,2026-05-09,2026-05-09T21:33:26Z,2026-05-09 14:34:15
1,PrizePicks,player_points,Tobias Harris,Under,21.5,-137,2026-05-09,2026-05-09T21:33:26Z,2026-05-09 14:34:15
2,PrizePicks,player_points,Duncan Robinson,Over,11.5,-137,2026-05-09,2026-05-09T21:33:26Z,2026-05-09 14:34:15
3,PrizePicks,player_points,Duncan Robinson,Under,11.5,-137,2026-05-09,2026-05-09T21:33:26Z,2026-05-09 14:34:15
4,PrizePicks,player_points,Shai Gilgeous-Alexander,Over,29.0,-137,2026-05-10,2026-05-09T21:34:00Z,2026-05-09 14:34:15


In [7]:
import json

prizepicks_path = "data/props/prizepicks/prizepicks_2026-05-09_172548.json"
with open(prizepicks_path) as f:
    data = json.load(f)

df = pd.DataFrame(data["projections"])
df.rename(columns={"player": "NAME", "stat_type": "CATEGORY", "line_score": "LINE", "odds_type": "ODDS_TYPE", 'updated_at': 'UPDATED_AT'}, inplace=True)
df = df[df["ODDS_TYPE"] == "standard"]
pts_lines = df[df["CATEGORY"] == "Points"]
ast_lines = df[df["CATEGORY"] == "Assists"]
reb_lines = df[df["CATEGORY"] == "Rebounds"]
pts_names = pts_lines["NAME"].unique()
ast_names = ast_lines["NAME"].unique()
reb_names = reb_lines["NAME"].unique()

### Load my models

In [8]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb_2026-05-07.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb_2026-05-07.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb_2026-05-07.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb_2026-05-08.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [9]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
pts_preds.head(10)

Team odds: NBA_20260509_143842.json
[SKIP] Kelly Oubre: min_pipeline returned None (need >= 10 games)
[SKIP] Terrence Shannon: min_pipeline returned None (need >= 10 games)
[SKIP] Kelly Oubre: min_pipeline returned None (need >= 10 games)


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,RATE_HISTORY,STAT_Q10,STAT_Q50,STAT_Q90
0,Shai Gilgeous-Alexander,PTS,29.39,32.60,39.08,0.5231,0.7613,0.9803,"[0.9400179051029544, 0.8468595624558927, 0.891...",15.37,24.81,38.31
1,Austin Reaves,PTS,33.35,38.83,40.87,0.3820,0.6066,0.8361,"[0.606826801517067, 0.6584723441615452, 0.6489...",12.74,23.55,34.17
2,LeBron James,PTS,33.08,37.92,41.09,0.3825,0.6188,0.8799,"[0.8722741433021807, 1.0876132930513596, 0.494...",12.65,23.46,36.16
3,Chet Holmgren,PTS,26.02,32.58,36.23,0.2609,0.5049,0.7706,"[0.3958529688972667, 0.9452363090772694, 0.681...",6.79,16.45,27.91
4,Ajay Mitchell,PTS,23.92,31.74,36.89,0.2336,0.4908,0.7455,"[0.6204756980351602, 0.3725716313314999, 0.419...",5.59,15.58,27.50
5,Rui Hachimura,PTS,30.56,37.09,41.27,0.1698,0.3681,0.6018,"[0.4224207961007311, 0.759493670886076, 0.3373...",5.19,13.65,24.84
6,Marcus Smart,PTS,27.91,34.57,38.38,0.1476,0.3688,0.5734,"[0.3324099722991689, 0.1901140684410646, 0.445...",4.12,12.75,22.01
7,Cason Wallace,PTS,19.42,27.10,31.85,0.1553,0.3467,0.5155,"[0.3776553894571203, 0.7285421567883434, 0.220...",3.02,9.40,16.42
8,Jared McCain,PTS,5.15,16.45,21.04,0.1301,0.4785,0.9162,"[0.946372239747634, 1.0434782608695652, 0.5901...",0.67,7.87,19.28
9,Luguentz Dort,PTS,19.38,25.81,32.42,0.0943,0.1733,0.4588,"[0.3539300988054859, 0.2995008319467554, 0.244...",1.83,4.47,14.88


In [10]:
from live import adjust_predictions

# Build contexts dict once (using the notebook's get_game_context)
game_contexts = {
    name: get_game_context(base_df, name, team_odds, is_playoff=True)
    for name in pts_preds["PLAYER_NAME"]
}

# Adjust the model's Q50 predictions with scenario signals
pts_preds = adjust_predictions(pts_preds, base_df, game_contexts)
ast_preds = adjust_predictions(ast_preds, base_df, game_contexts)
reb_preds = adjust_predictions(reb_preds, base_df, game_contexts)
pts_preds.head()

Shai Gilgeous-Alexander [PTS] [ix: away_low_pace]  pace_bucket=low_pace  MIN: 32.6→34.1 (Δ+1.54)  RATE: 0.7613→0.5732 (Δ-0.1881)
Austin Reaves [PTS] [ix: home_low_pace]  pace_bucket=low_pace  MIN: 38.8→40.0 (Δ+1.12)  RATE: 0.6066→0.3331 (Δ-0.2735)
LeBron James [PTS] [ix: home_low_pace]  pace_bucket=low_pace  MIN: 37.9→40.4 (Δ+2.46)  RATE: 0.6188→0.5193 (Δ-0.0995)
Chet Holmgren [PTS] [ix: away_low_pace]  pace_bucket=low_pace  MIN: 32.6→32.6 (Δ+0.02)  RATE: 0.5049→0.4723 (Δ-0.0326)
Ajay Mitchell [PTS]  pace_bucket=low_pace  MIN: 31.7→36.7 (Δ+5.00)  RATE: 0.4908→0.5411 (Δ+0.0503)
Rui Hachimura [PTS] [ix: home_low_pace]  pace_bucket=low_pace  MIN: 37.1→42.1 (Δ+5.00)  RATE: 0.3681→0.3813 (Δ+0.0132)
Marcus Smart [PTS]  pace_bucket=low_pace  MIN: 34.6→38.3 (Δ+3.73)  RATE: 0.3688→0.3993 (Δ+0.0305)
Cason Wallace [PTS] [ix: away_low_pace]  pace_bucket=low_pace  MIN: 27.1→25.2 (Δ-1.93)  RATE: 0.3467→0.3154 (Δ-0.0313)
Jared McCain [PTS]  pace_bucket=low_pace  MIN: 16.4→20.5 (Δ+4.00)  RATE: 0.4785→

,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,RATE_HISTORY,STAT_Q10,STAT_Q50,STAT_Q90,ADJ_CONTEXT_OK,ADJ_CONTEXT_ERR,ADJ_ACTIVE_STARS,ADJ_STARS_MISSING,ADJ_SPREAD_ROLE,ADJ_CTX_SPREAD,ADJ_CTX_TOTAL,ADJ_MIN_DELTA,ADJ_RATE_DELTA,ADJ_MIN_SHIFT,ADJ_RATE_SHIFT,ADJ_USED_INTERACTION,ADJ_MIN_LOG,ADJ_RATE_LOG
0,Shai Gilgeous-Alexander,PTS,30.93,34.14,40.62,0.3350,0.5732,0.7922,"[0.751918, 0.65876, 0.703642, 0.472329, 0.6685...",10.36,19.57,32.18,True,None,3,0,favorite,-9.5,210.5,1.5368,-0.1881,1.54,-0.1881,True,"{'stars': (0.3413, 77), 'pace': (0.0, 55), 'in...","{'stars': (-0.0287, 77), 'pace': (-0.0528, 55)..."
1,Austin Reaves,PTS,34.47,39.95,41.99,0.1085,0.3331,0.5626,"[0.333327, 0.384972, 0.375418, 0.424602, 0.243...",3.74,13.31,23.62,True,None,3,0,underdog,9.5,210.5,1.1234,-0.2735,1.12,-0.2735,True,"{'stars': (0.6612, 53), 'pace': (-0.0197, 42),...","{'stars': (-0.0436, 53), 'pace': (-0.091, 42),..."
2,LeBron James,PTS,35.54,40.38,43.55,0.2830,0.5193,0.7804,"[0.772774, 0.988113, 0.394884, 0.614786, 0.543...",10.06,20.97,33.99,True,None,3,0,underdog,9.5,210.5,2.4633,-0.0995,2.46,-0.0995,True,"{'stars': (-0.184, 54), 'pace': (0.6048, 47), ...","{'stars': (-0.0703, 54), 'pace': (-0.007, 47),..."
3,Chet Holmgren,PTS,26.04,32.60,36.25,0.2283,0.4723,0.7380,"[0.363253, 0.912636, 0.649218, 0.927912, 0.612...",5.94,15.40,26.75,True,None,3,0,favorite,-9.5,210.5,0.0163,-0.0326,0.02,-0.0326,True,"{'stars': (-0.2404, 76), 'pace': (-0.1633, 42)...","{'stars': (-0.002, 76), 'pace': (0.0095, 42), ..."
4,Ajay Mitchell,PTS,28.92,36.74,41.89,0.2839,0.5411,0.7958,"[0.670776, 0.422872, 0.469946, 0.339158, 0.463...",8.21,19.88,33.34,True,None,3,0,favorite,-9.5,210.5,5.0000,0.0503,5.00,0.0503,False,"{'stars': (-0.4592, 30), 'pace': (-0.0894, 28)...","{'stars': (-0.0073, 30), 'pace': (-0.0261, 28)..."


### Get Line Probabilities

In [11]:
lines_dfs_pts = pts_lines
lines_dfs_ast = ast_lines
lines_dfs_reb = reb_lines


all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
80,Ayo Dosunmu,PTS,11.5,25.62,33.04,37.47,3.02,11.02,23.52,0.509,0.491
73,Anthony Edwards,PTS,25.5,23.67,33.28,39.29,5.27,16.52,29.01,0.077,0.923
67,OG Anunoby,PTS,14.5,33.72,39.54,42.00,9.12,19.37,30.08,0.857,0.143
21,Luke Kennard,REB,2.5,17.01,23.68,31.24,0.54,2.56,7.75,0.547,0.453
74,De'Aaron Fox,PTS,17.5,28.90,34.22,38.02,5.83,13.91,26.75,0.312,0.688


In [17]:
import json

pinnacle_path = "data/props/pinnacle/pinnacle_2026-05-09_172549.json"
with open(pinnacle_path) as f:
    data = json.load(f)

rows = []
for game in data["games"]:
    for prop in game.get("props", []):
        rows.append({
            "matchup_id": game["matchup_id"],
            "game": game["label"],
            **prop,
        })

df = pd.DataFrame(rows)
df.rename(columns={"player": "NAME", "stat": "CATEGORY", "line": "LINE", "decimal_over": "DECIMAL_OVER", "decimal_under": "DECIMAL_UNDER", "american_over": "AMERICAN_OVER", "american_under": "AMERICAN_UNDER"}, inplace=True)
market_map = {"PTS": "points", "REB": "rebounds", "AST": "assists"}
BOOK_CATEGORIES = tuple(market_map.values())

inv_over = 1 / df["DECIMAL_OVER"]
inv_under = 1 / df["DECIMAL_UNDER"]
z = inv_over + inv_under
df["NO_VIG_IMPLIED_OVER"] = (inv_over / z).round(3)
df["NO_VIG_IMPLIED_UNDER"] = (inv_under / z).round(3)
df = df[df["CATEGORY"].isin(BOOK_CATEGORIES)].copy()
df["LINE"] = df["LINE"].astype(float)

all_line_probs["CATEGORY_KEY"] = all_line_probs["MARKET"].replace(market_map)
all_line_probs["LINE"] = all_line_probs["LINE"].astype(float)

In [24]:
bet365_path = "data/props/365+mgm_props/365+mgm_20260510_002605.json"
with open(bet365_path) as f:
    data = json.load(f)

records = pd.DataFrame(data["records"])
records.head()

# Example: First Basket only, one bookmaker
records[records["BOOKMAKER"] == "Bet365"]

,NAME,MARKET,LINE,OVER,UNDER,BOOKMAKER,EVENT_ID,HOME,AWAY,START


In [18]:
# Exact merge only: player + stat category + numerical line matches Pinnacle
merged_overlap = all_line_probs.merge(
    df,
    left_on=["PLAYER_NAME", "CATEGORY_KEY", "LINE"],
    right_on=["NAME", "CATEGORY", "LINE"],
    how="inner",
    suffixes=("_model", ""),
)
merged_overlap["line_match_gap"] = 0.0
merged_overlap["line_match_kind"] = "exact"

merged_overlap["edge_over"] = merged_overlap["P_OVER"] - merged_overlap["NO_VIG_IMPLIED_OVER"]
merged_overlap["edge_under"] = merged_overlap["P_UNDER"] - merged_overlap["NO_VIG_IMPLIED_UNDER"]


merged_overlap["favors_over"] = merged_overlap["P_OVER"] >= merged_overlap["P_UNDER"]
merged_overlap["fair_when_favored"] = np.where(
    merged_overlap["favors_over"],
    merged_overlap["NO_VIG_IMPLIED_OVER"],
    merged_overlap["NO_VIG_IMPLIED_UNDER"],
)
merged_overlap["edge_on_favored"] = np.where(
    merged_overlap["favors_over"],
    merged_overlap["edge_over"],
    merged_overlap["edge_under"],
)

merged_overlap["implied_favors_over"] = (
    merged_overlap["NO_VIG_IMPLIED_OVER"] > merged_overlap["NO_VIG_IMPLIED_UNDER"]
)
merged_overlap["both_same_side"] = (
    merged_overlap["favors_over"] == merged_overlap["implied_favors_over"]
)

#Trading filter: keep only props where model lean matches Pinnacle no-vig lean (drop disagreement).
props_agrees_pinnacle_implied = merged_overlap.loc[merged_overlap["both_same_side"]].copy()

# Sharp filter for leg-building: exact Pinnacle line + model agrees with Pinnacle no-vig lean.
# Use `line_probs_for_legs` (not full `all_line_probs`) before ranking/selecting legs.
line_probs_for_legs = props_agrees_pinnacle_implied.copy()

n_pinnacle = len(df)
n_merged = len(merged_overlap)
n_same_side = len(props_agrees_pinnacle_implied)
n_drop_disagree = n_merged - n_same_side
pct_overlap = 100.0 * n_merged / n_pinnacle if n_pinnacle else 0.0
pct_merged = 100.0 * n_same_side / n_merged if n_merged else 0.0
pct_vs_pinnacle = 100.0 * n_same_side / n_pinnacle if n_pinnacle else 0.0

print(f"Pinnacle PTS/REB/AST offerings (this snapshot): {n_pinnacle}")
print(f"overlap (model ∩ Pinnacle, same player/stat/line): {n_merged} / {n_pinnacle} ({pct_overlap:.1f}% of Pinnacle)")
print(f"dropped disagreement (model lean ≠ implied lean): {n_drop_disagree} / {n_merged}")
print(f"kept aligned props: {n_same_side} / {n_merged} ({pct_merged:.1f}% of overlap) | {pct_vs_pinnacle:.1f}% of Pinnacle lines")

model_prop_markets = ("PTS", "REB", "AST")
n_model_props = int(all_line_probs["MARKET"].isin(model_prop_markets).sum())
if n_model_props:
    pct_sheet = 100.0 * n_same_side / n_model_props
    print(
        f"aligned vs model PTS/REB/AST sheet: {n_same_side} / {n_model_props} ({pct_sheet:.1f}% — omits unmatched Pinnacle lines)"
    )

# Rows in all_line_probs (by MARKET) whose exact line matched Pinnacle
_probs_pts_only = all_line_probs[all_line_probs["MARKET"].isin(model_prop_markets)].copy()
_matched_model_keys = merged_overlap[["PLAYER_NAME", "CATEGORY_KEY", "LINE"]].drop_duplicates()
_matched_in_probs = _probs_pts_only.merge(_matched_model_keys, on=["PLAYER_NAME", "CATEGORY_KEY", "LINE"], how="inner")
_tot = _probs_pts_only.groupby("MARKET").size().rename("lines_in_model")
_hit = _matched_in_probs.groupby("MARKET").size().rename("matched_pinnacle")
match_by_category = (
    pd.concat([_tot, _hit], axis=1).fillna(0).astype({"matched_pinnacle": int})
)
match_by_category["lines_in_model"] = match_by_category["lines_in_model"].astype(int)
match_by_category["pct_matched"] = (
    100.0 * match_by_category["matched_pinnacle"] / match_by_category["lines_in_model"]
).where(match_by_category["lines_in_model"] > 0, 0.0)

print("\nLines in all_line_probs with an exact Pinnacle line match (by MARKET):")
print(match_by_category.round(1).to_string())

show_cols = [
    c
    for c in [
        "PLAYER_NAME",
        "game",
        "LINE",
        "line_match_gap",
        "line_match_kind",
        "MARKET",
        "P_OVER",
        "P_UNDER",
        "NO_VIG_IMPLIED_OVER",
        "NO_VIG_IMPLIED_UNDER",
        "favors_over",
        "implied_favors_over",
        "edge_over",
        "edge_under",
    ]
    if c in line_probs_for_legs.columns
]
line_probs_for_legs[show_cols]

Pinnacle PTS/REB/AST offerings (this snapshot): 101
overlap (model ∩ Pinnacle, same player/stat/line): 42 / 101 (41.6% of Pinnacle)
dropped disagreement (model lean ≠ implied lean): 18 / 42
kept aligned props: 24 / 42 (57.1% of overlap) | 23.8% of Pinnacle lines
aligned vs model PTS/REB/AST sheet: 24 / 97 (24.7% — omits unmatched Pinnacle lines)

Lines in all_line_probs with an exact Pinnacle line match (by MARKET):
        lines_in_model  matched_pinnacle  pct_matched
MARKET                                               
AST                 15                 7         46.7
PTS                 50                23         46.0
REB                 32                12         37.5


,PLAYER_NAME,game,LINE,line_match_gap,line_match_kind,MARKET,P_OVER,P_UNDER,NO_VIG_IMPLIED_OVER,NO_VIG_IMPLIED_UNDER,favors_over,implied_favors_over,edge_over,edge_under
0,LeBron James,Oklahoma City Thunder vs Los Angeles Lakers,7.5,0.0,exact,AST,0.406,0.594,0.472,0.528,False,False,-0.066,0.066
1,Shai Gilgeous-Alexander,Oklahoma City Thunder vs Los Angeles Lakers,6.5,0.0,exact,AST,0.711,0.289,0.518,0.482,True,True,0.193,-0.193
2,Chet Holmgren,Oklahoma City Thunder vs Los Angeles Lakers,1.5,0.0,exact,AST,0.494,0.506,0.489,0.511,False,False,0.005,-0.005
7,LeBron James,Oklahoma City Thunder vs Los Angeles Lakers,6.5,0.0,exact,REB,0.459,0.541,0.453,0.547,False,False,0.006,-0.006
10,Joel Embiid,New York Knicks vs Philadelphia 76Ers,7.5,0.0,exact,REB,0.585,0.415,0.529,0.471,True,True,0.056,-0.056
13,Naz Reid,San Antonio Spurs vs Minnesota Timberwolves,6.5,0.0,exact,REB,0.776,0.224,0.515,0.485,True,True,0.261,-0.261
15,Anthony Edwards,San Antonio Spurs vs Minnesota Timberwolves,5.5,0.0,exact,REB,0.706,0.294,0.528,0.472,True,True,0.178,-0.178
16,De'Aaron Fox,San Antonio Spurs vs Minnesota Timberwolves,3.5,0.0,exact,REB,0.276,0.724,0.500,0.500,False,False,-0.224,0.224
17,Dylan Harper,San Antonio Spurs vs Minnesota Timberwolves,3.5,0.0,exact,REB,0.483,0.517,0.500,0.500,False,False,-0.017,0.017
18,Keldon Johnson,San Antonio Spurs vs Minnesota Timberwolves,3.5,0.0,exact,REB,0.535,0.465,0.510,0.490,True,True,0.025,-0.025


In [19]:
# Props where model lean agrees with Pinnacle no-vig implied lean (exact-line overlap only).
# Requires the previous cell defining `merged_overlap` with columns MARKET / both_same_side / etc.

agreement = merged_overlap.loc[merged_overlap["both_same_side"]].copy()

_overlap_counts = merged_overlap.groupby("MARKET", observed=True).agg(
    n_overlap=("PLAYER_NAME", "count"),
)
_agree_counts = agreement.groupby("MARKET", observed=True).agg(
    n_agree=("PLAYER_NAME", "count"),
)
by_market_counts = (
    _overlap_counts.join(_agree_counts, how="outer")
    .fillna(0)
    .astype({"n_overlap": int, "n_agree": int})
)
by_market_counts["pct_agree_of_overlap"] = (
    100.0
    * by_market_counts["n_agree"]
    / by_market_counts["n_overlap"].replace(0, np.nan)
).round(1)
by_market_counts = by_market_counts.sort_values("n_agree", ascending=False)

_preview_cols = [
    c for c in [
        "PLAYER_NAME",
        "game",
        "MARKET",
        "LINE",
        "P_OVER",
        "P_UNDER",
        "NO_VIG_IMPLIED_OVER",
        "NO_VIG_IMPLIED_UNDER",
        "favors_over",
        "implied_favors_over",
    ]
    if c in agreement.columns
]

print("Agreements by MARKET (with overlap denominator)")
print(by_market_counts.to_string())

agreement[_preview_cols].head(50)



Agreements by MARKET (with overlap denominator)
        n_overlap  n_agree  pct_agree_of_overlap
MARKET                                          
PTS            23       14                  60.9
REB            12        7                  58.3
AST             7        3                  42.9


,PLAYER_NAME,game,MARKET,LINE,P_OVER,P_UNDER,NO_VIG_IMPLIED_OVER,NO_VIG_IMPLIED_UNDER,favors_over,implied_favors_over
0,LeBron James,Oklahoma City Thunder vs Los Angeles Lakers,AST,7.5,0.406,0.594,0.472,0.528,False,False
1,Shai Gilgeous-Alexander,Oklahoma City Thunder vs Los Angeles Lakers,AST,6.5,0.711,0.289,0.518,0.482,True,True
2,Chet Holmgren,Oklahoma City Thunder vs Los Angeles Lakers,AST,1.5,0.494,0.506,0.489,0.511,False,False
7,LeBron James,Oklahoma City Thunder vs Los Angeles Lakers,REB,6.5,0.459,0.541,0.453,0.547,False,False
10,Joel Embiid,New York Knicks vs Philadelphia 76Ers,REB,7.5,0.585,0.415,0.529,0.471,True,True
13,Naz Reid,San Antonio Spurs vs Minnesota Timberwolves,REB,6.5,0.776,0.224,0.515,0.485,True,True
15,Anthony Edwards,San Antonio Spurs vs Minnesota Timberwolves,REB,5.5,0.706,0.294,0.528,0.472,True,True
16,De'Aaron Fox,San Antonio Spurs vs Minnesota Timberwolves,REB,3.5,0.276,0.724,0.500,0.500,False,False
17,Dylan Harper,San Antonio Spurs vs Minnesota Timberwolves,REB,3.5,0.483,0.517,0.500,0.500,False,False
18,Keldon Johnson,San Antonio Spurs vs Minnesota Timberwolves,REB,3.5,0.535,0.465,0.510,0.490,True,True
